# GRPO Loss

**GRPO: Group Relative Policy Optimization**

来自 DeepSeek《DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models》(arXiv:2402.03300)

## 核心思想：用组内相对排名代替 Critic 网络

PPO 需要额外训练一个与策略模型同等规模的 **Value Network（Critic）** 来估计优势 $\hat{A}_t$，
显存占用和计算开销近乎翻倍，且 Critic 本身的训练稳定性也是难题。

GRPO 的做法：对同一个 prompt $q$ 采样 **一组** 响应 $\{o_1, o_2, \ldots, o_G\}$，
直接用组内奖励的 **z-score** 作为优势估计，从而 **完全去掉 Critic**。

```
        PPO                              GRPO
   ┌──────────┐                    ┌──────────┐
   │  Policy  │──┐                 │  Policy  │──┐  采样 G 条响应
   └──────────┘  │ 需要 A_t         └──────────┘  │
   ┌──────────┐  │                                │  o_1 → r_1  ┐
   │  Critic  │──┘  ← 额外一个大模型                 └→ o_2 → r_2  ├→ z-score → A_i
   └──────────┘                                       ...        │
                                                      o_G → r_G  ┘
```

**直觉**：不需要知道「这条响应的绝对价值是多少」，只需要知道「它比同组的兄弟们好还是差」。
比同组平均好 → 提高它的概率；比平均差 → 降低它的概率。

## GRPO 目标函数

$$
\mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \left[ \min \left( \frac{\pi_{\theta}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} | q, o_{i,<t})} \hat{A}_{i,t}, \text{clip} \left( \frac{\pi_{\theta}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} | q, o_{i,<t})}, 1-\epsilon, 1+\epsilon \right) \hat{A}_{i,t} \right) - \beta \mathbb{D}_{\text{KL}}[\pi_{\theta} || \pi_{\text{ref}}] \right]
$$

逐项拆解：

| 记号 | 含义 |
|------|------|
| $G$ | 组大小（同一 prompt 采样的响应条数），典型值 8 / 16 / 64 |
| $\lvert o_i \rvert$ | 第 $i$ 条响应的 token 数（**每条各不相同**） |
| $\frac{1}{G}\sum_i \frac{1}{\lvert o_i \rvert}\sum_t$ | 先在序列内对 token 取均值，再在组内对序列取均值 |
| $r_{i,t}(\theta) = \frac{\pi_\theta}{\pi_{\theta_{\text{old}}}}$ | 重要性采样比率，修正「用旧策略采的样训新策略」的分布偏差 |
| $\text{clip}(\cdot, 1-\epsilon, 1+\epsilon)$ | 信任域约束，防止单步更新走太远，典型 $\epsilon = 0.2$ |
| $\min(\cdot, \cdot)$ | 取悲观下界，保证目标是真实目标的 lower bound |
| $\beta \mathbb{D}_{\text{KL}}$ | 拉住策略不要偏离 SFT 参考模型太远，典型 $\beta = 0.001 \sim 0.04$ |
| 最外层负号 | 把「最大化目标 $\mathcal{J}$」转成「最小化损失 $\mathcal{L}$」 |

### KL 散度项（k3 估计量）

$$
\mathbb{D}_{\text{KL}}[\pi_{\theta} || \pi_{\text{ref}}] = \frac{\pi_{\text{ref}}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta}(o_{i,t} | q, o_{i,<t})} - \log \frac{\pi_{\text{ref}}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta}(o_{i,t} | q, o_{i,<t})} - 1
$$

### 组相对优势

$$
\hat{A}_{i,t} = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}
$$

注意下标 $t$：GRPO 中同一条响应的**所有 token 共享同一个优势值**（outcome supervision），
不像 PPO 那样每个 token 有独立的 $\hat{A}_t$。

## GRPO vs PPO 速查

| 维度 | PPO | GRPO |
|------|-----|------|
| 优势估计 | Critic 网络 + GAE | 组内奖励 z-score |
| 额外模型 | Value Network（同等规模） | 无 |
| 显存开销 | 4 份模型（policy/ref/reward/critic） | 3 份（policy/ref/reward） |
| 优势粒度 | 每 token 独立 $\hat{A}_t$ | 整条响应共享 $\hat{A}_i$ |
| KL 惩罚位置 | 混进 reward 里 | **直接加在 loss 上** |
| 采样要求 | 每 prompt 1 条即可 | 每 prompt 必须 $G$ 条 |

> **KL 位置的差异很关键**：PPO 把 KL 惩罚塞进 reward（$r_t - \beta\log\frac{\pi_\theta}{\pi_{\text{ref}}}$），
> 于是 KL 会经过 GAE 累积传播；GRPO 直接把 KL 作为 loss 的一项，梯度路径更短更直接。

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

---
## 1. 组相对优势（Group Relative Advantage）

In [ ]:
def grpo_advantage(rewards, eps=1e-5):
    """
    计算组相对优势：把组内奖励标准化为均值 0、标准差 1 的 z-score

        A_i = (r_i - mean(r)) / (std(r) + eps)

    A_i > 0：该响应优于组内平均 → 训练时提升其生成概率
    A_i < 0：该响应劣于组内平均 → 训练时降低其生成概率

    这就是 GRPO 不需要 Critic 的原因：优势不再由 Value Network 预测，
    而是直接由「同组兄弟样本」的相对排名给出。

    Args:
        rewards: Tensor[G]，组内 G 条响应各自的标量奖励
        eps:     防止 std=0 时除零（组内奖励全相同时会发生）
    Returns:
        Tensor[G]，归一化后的优势值

    注：torch.std 默认使用无偏估计（分母 n-1），而论文写的是总体标准差（分母 n）。
        G 较大时差异可忽略；若要严格对齐论文可用 rewards.std(unbiased=False)。
        主流实现（verl / TRL）沿用默认的无偏估计。
    """
    return (rewards - rewards.mean()) / (rewards.std() + eps)


# ---- 直观验证 ----
demo = torch.tensor([1., 1., 0., 0., 0., 0., 0., 0.])
print(f"奖励 {demo.tolist()}")
print(f"优势 {[round(v, 3) for v in grpo_advantage(demo).tolist()]}")
print(f"优势和 = {grpo_advantage(demo).sum():.2e}   ← z-score 的均值恒为 0\n")

# 退化情形：组内全对或全错时优势恒为 0，该组不产生任何梯度
for r in [torch.ones(8), torch.zeros(8)]:
    print(f"奖励全为 {r[0].item():.0f} → 优势 {grpo_advantage(r).tolist()}  (梯度为零，样本被浪费)")
print("\n↑ 这正是 DAPO 提出 Dynamic Sampling 要解决的问题。")

---
## 2. KL 散度：为什么用 k3 估计量

我们要算的是 $\mathbb{D}_{\text{KL}}[\pi_\theta \| \pi_{\text{ref}}] = \mathbb{E}_{x \sim \pi_\theta}\left[\log\frac{\pi_\theta(x)}{\pi_{\text{ref}}(x)}\right]$。

在 RLHF 里我们只有**从 $\pi_\theta$ 采样出的那几个 token**，需要用蒙特卡洛估计。
记 $r = \frac{\pi_{\text{ref}}(x)}{\pi_\theta(x)}$（注意是 **ref 在分子**），John Schulman 给出三种估计量：

| 估计量 | 表达式 | 无偏 | 恒非负 | 方差（小 KL 时） |
|--------|--------|------|--------|------|
| k1 | $-\log r$ | ✅ | ❌ | 高 |
| k2 | $\frac{1}{2}(\log r)^2$ | ❌ | ✅ | 低 |
| **k3** | $r - \log r - 1$ | ✅ | ✅ | 低 |

**GRPO 采用 k3**，因为它同时满足无偏和非负——单个样本的估计值也永远 $\geq 0$，
符合 KL 散度的数学性质，训练日志里不会出现「负 KL」这种看起来就不对的东西。

> 方差那一列标注了「小 KL 时」：k3 含 $e^{\log r}$ 项，当两个分布**离得很远**时这一项会爆炸，
> 方差会被 k2 反超。但 RLHF 的 KL 惩罚正是为了把 $\pi_\theta$ 拉在 $\pi_{\text{ref}}$ 附近，
> 实际工作区间的 KL 只有 $10^{-4} \sim 10^{-2}$ 量级；在这个区间 k3 的方差已低到与 k2 相当，
> 却不像 k2 那样有偏（下方有扫描验证）。

### 落到 log 空间

代码里拿到的是 log prob，所以：

$$\log r = \log\pi_{\text{ref}} - \log\pi_\theta \quad\Longrightarrow\quad k_3 = \underbrace{e^{(\log\pi_{\text{ref}} - \log\pi_\theta)}}_{r} - (\log\pi_{\text{ref}} - \log\pi_\theta) - 1$$

> ⚠️ **易错点**：第一项是 $e^{\log\pi_{\text{ref}} - \log\pi_\theta}$，**不是** $e^{\log\pi_\theta}$。
> 写错会导致 KL 出现负值、且 $\pi_\theta = \pi_{\text{ref}}$ 时不为零，下方有验证。
> 另外不要写成 `ref.exp() / pi.exp()`——数学上等价，但 log prob 很负时两个 exp 都下溢到 0，得到 `nan`。

In [ ]:
def grpo_kl(pi_logprob, pi_ref_logprob):
    """
    k3 KL 散度估计量（GRPO / DeepSeekMath 采用）

        D_KL[pi_theta || pi_ref] ≈ r - log(r) - 1,    r = pi_ref / pi_theta

    在 log 空间实现，避免 exp 下溢：
        log_r = log(pi_ref) - log(pi_theta)
        k3    = exp(log_r) - log_r - 1

    性质：恒 >= 0，且当 pi_theta == pi_ref 时恰为 0。

    Args:
        pi_logprob:     当前策略对已采样 token 的 log 概率，[G, seq_len]
        pi_ref_logprob: 参考（SFT）模型对同一 token 的 log 概率，[G, seq_len]
    Returns:
        Tensor，逐 token 的 KL 估计，形状同输入
    """
    log_r = pi_ref_logprob - pi_logprob      # log(pi_ref / pi_theta)
    return torch.exp(log_r) - log_r - 1.0


# ══════════ 验证 1：非负性 & 同分布归零 ══════════
lp = torch.linspace(-8, -0.1, 60)
gp, gr = torch.meshgrid(lp, lp, indexing="ij")
kl_grid = grpo_kl(gp, gr)
print(f"[非负性]   3600 组取值中 min(KL) = {kl_grid.min():.3e}   (必须 >= 0)")

same = grpo_kl(torch.tensor(0.3).log(), torch.tensor(0.3).log())
print(f"[同分布]   KL(p || p) = {same:.3e}   (必须 == 0)")
assert kl_grid.min() >= -1e-6 and abs(same) < 1e-6
print("✅ 通过\n")

# ══════════ 验证 2：k3 确实是真实 KL 的无偏估计 ══════════
# 词表规模小的时候可以算解析 KL，用它当 ground truth 检验采样估计
torch.manual_seed(0)
V = 50
pi_lp  = F.log_softmax(torch.randn(V), dim=-1)
ref_lp = F.log_softmax(torch.randn(V) * 0.5, dim=-1)

true_kl = (pi_lp.exp() * (pi_lp - ref_lp)).sum()                 # 解析解：遍历整个词表
idx     = torch.multinomial(pi_lp.exp(), 200_000, replacement=True)   # 从 pi_theta 采样
k3_mc   = grpo_kl(pi_lp[idx], ref_lp[idx]).mean()                # k3 蒙特卡洛估计

print(f"[无偏性]   解析 KL      = {true_kl:.6f}")
print(f"           k3 采样均值  = {k3_mc:.6f}")
print(f"           相对误差     = {abs(k3_mc - true_kl) / true_kl * 100:.2f}%   (20 万次采样)")
assert abs(k3_mc - true_kl) / true_kl < 0.05
print("✅ 通过")

In [ ]:
# ══════════ k1 / k2 / k3 三种估计量对比 ══════════
# 说明为什么 GRPO 选 k3：无偏 + 单样本恒非负 + 小 KL 区间方差最低

def kl_k1(pi_lp, ref_lp):
    """无偏，但单个样本可能为负，方差大"""
    return -(ref_lp - pi_lp)

def kl_k2(pi_lp, ref_lp):
    """恒非负、方差小，但有偏"""
    return 0.5 * (ref_lp - pi_lp) ** 2

def kl_k3(pi_lp, ref_lp):
    """无偏 + 恒非负 —— GRPO 的选择"""
    log_r = ref_lp - pi_lp
    return torch.exp(log_r) - log_r - 1.0


torch.manual_seed(0)
V, N = 50, 200_000
base = torch.randn(V)
ESTIMATORS = [("k1", kl_k1), ("k2", kl_k2), ("k3", kl_k3)]

# 扫描不同的「策略-参考模型」距离：scale 越大两个分布离得越远
print("偏差对比（采样均值 - 解析 KL，越接近 0 越无偏）")
print(f"{'扰动尺度':>8} {'解析KL':>9} | {'k1 偏差':>10} {'k2 偏差':>10} {'k3 偏差':>10}")
print("-" * 56)
rows = []
for scale in [0.02, 0.05, 0.1, 0.3, 0.5, 1.0]:
    pi_lp  = F.log_softmax(base, dim=-1)
    ref_lp = F.log_softmax(base + torch.randn(V) * scale, dim=-1)
    true_kl = (pi_lp.exp() * (pi_lp - ref_lp)).sum()              # 解析解（遍历词表）
    idx = torch.multinomial(pi_lp.exp(), N, replacement=True)     # 从 pi_theta 采样
    vals = {n: f(pi_lp[idx], ref_lp[idx]) for n, f in ESTIMATORS}
    rows.append((scale, true_kl, vals))
    print(f"{scale:>8.2f} {true_kl:>9.5f} | "
          + " ".join(f"{vals[n].mean() - true_kl:>+10.5f}" for n, _ in ESTIMATORS))

print("\n方差对比（标准差，越小估计越稳定）")
print(f"{'扰动尺度':>8} {'解析KL':>9} | {'k1 std':>9} {'k2 std':>9} {'k3 std':>9} | 最优")
print("-" * 62)
for scale, true_kl, vals in rows:
    stds = {n: vals[n].std().item() for n, _ in ESTIMATORS}
    best = min(stds, key=stds.get)
    print(f"{scale:>8.2f} {true_kl:>9.5f} | "
          + " ".join(f"{stds[n]:>9.4f}" for n, _ in ESTIMATORS) + f" | {best}")

print("\n非负性检查（单样本最小值，扰动尺度 0.3）")
pi_lp  = F.log_softmax(base, dim=-1)
ref_lp = F.log_softmax(base + torch.randn(V) * 0.3, dim=-1)
idx = torch.multinomial(pi_lp.exp(), N, replacement=True)
for n, f in ESTIMATORS:
    v = f(pi_lp[idx], ref_lp[idx])
    print(f"  {n}: min = {v.min():>9.4f}   {'✅ 恒非负' if v.min() >= 0 else '❌ 出现负值'}")

print("""
结论（对照上面两张表）：
  · k1  无偏，但单样本会出现负值，且方差在所有尺度下都最大（比 k2/k3 高 1~2 个数量级）
  · k2  恒非负、方差低，但有偏 —— 扰动越大偏差越明显（scale=1.0 时偏差高达 +0.166）
  · k3  同时拿到三项：无偏（偏差始终 ~1e-5）+ 恒非负 + 小 KL 区间方差与 k2 持平

  RLHF 中 KL 惩罚的作用就是把 pi_theta 拉在 pi_ref 附近，实际工作在表格上半部分
  （KL ~ 1e-4 到 3e-2）。在这个区间 k3 的方差已低到与 k2 相当，却不像 k2 那样有偏，
  也不像 k1 那样会算出负 KL —— 这就是 GRPO 选 k3 的理由。

  注意 scale=0.3~0.5 时 k3 的 exp 项开始放大方差、被 k2 反超。
  这不影响实际使用：训练中 KL 一旦涨到这个量级，说明策略已经跑飞了。""")

---
## 3. 从 logits 取出 token 的 log prob：**必须做 shift 对齐**

这是 causal LM 训练里最经典的一个坑。

自回归模型在位置 $t$ 输出的分布，预测的是 **位置 $t+1$ 的 token**：

```
输入 tokens :   x_0    x_1    x_2    x_3    x_4
                 │      │      │      │      │
                 ▼      ▼      ▼      ▼      ▼
logits 位置 :    0      1      2      3      4
                 │      │      │      │      │
                 └─预测→ x_1   └─预测→ x_3   （位置1预测x_2，位置3预测x_4）
                        x_2           x_4
```

所以要取 $\log \pi(x_t \mid x_{<t})$，必须用 **位置 $t-1$ 的 logits** 去 gather **token $x_t$**：

```python
logprob = F.log_softmax(logits[:, :-1], dim=-1)   # 丢掉最后一个位置（它预测的 token 不存在）
target  = token_ids[:, 1:]                        # 丢掉第一个 token（它没有前文可预测）
```

> ⚠️ 若不做 shift，直接 `gather(log_softmax(logits), index=token_ids)`，
> 取到的是 $\log \pi(x_t \mid x_{\leq t})$——**用 $x_t$ 自己预测 $x_t$**，语义完全错误。
> 由于形状仍然合法、也不会报错，这个 bug 极易被忽略。

In [ ]:
def gather_token_logprob(logits, token_ids):
    """
    从 logits 中取出「实际采样到的 token」的 log 概率，并完成 causal LM 的 shift 对齐

    Args:
        logits:    模型原始输出，[bs, seq_len, vocab_size]
        token_ids: 实际的 token 序列，[bs, seq_len]
    Returns:
        Tensor[bs, seq_len - 1]，第 j 列对应 log P(token_ids[:, j+1] | 前文)

    注意返回长度是 seq_len - 1，后续构建 mask 时必须与之对齐。
    """
    logprob = F.log_softmax(logits[:, :-1], dim=-1)          # 去掉最后一个位置
    target  = token_ids[:, 1:].unsqueeze(-1)                 # 去掉第一个 token
    return torch.gather(logprob, dim=-1, index=target).squeeze(-1)


# ══════════ 验证：shift 后的结果与手工计算一致 ══════════
torch.manual_seed(0)
logits    = torch.randn(2, 6, 32)
token_ids = torch.randint(0, 32, (2, 6))

shifted = gather_token_logprob(logits, token_ids)
print(f"logits {tuple(logits.shape)} + tokens {tuple(token_ids.shape)} → logprob {tuple(shifted.shape)}")

# 手工核对：位置 2 的分布，预测 token 3
manual = F.log_softmax(logits[0, 2], dim=-1)[token_ids[0, 3]]
print(f"shifted[0, 2] = {shifted[0, 2]:.6f}")
print(f"手工计算      = {manual:.6f}   ← 位置2的分布 gather token3")
assert torch.allclose(shifted[0, 2], manual)
print("✅ 对齐正确\n")

# ══════════ 对比：不 shift 会拿到什么 ══════════
wrong = torch.gather(F.log_softmax(logits, -1), -1, token_ids.unsqueeze(-1)).squeeze(-1)
print(f"未 shift 版本 shape = {tuple(wrong.shape)}  ← 形状合法，不会报错，所以 bug 很隐蔽")
print(f"未 shift wrong[0,3] = {wrong[0, 3]:.6f}  (= 位置3的分布 gather token3，用自己预测自己)")
print(f"正确   shifted[0,2] = {shifted[0, 2]:.6f}  (= 位置2的分布 gather token3)")
print(f"两者数值不同: {not torch.allclose(wrong[0, 3], shifted[0, 2])}")

---
## 4. GRPO Loss 核心实现

### 关于 $\lvert o_i \rvert$ 的处理

公式里的 $\frac{1}{\lvert o_i \rvert}$ 是**每条响应各自的长度**。真实场景中同组响应长度差异很大
（有的 100 token，有的 2000 token），绝不能用一个标量长度广播到整个 batch。

正确做法是从 `completion_mask` 直接求和得到：`len_oi = completion_mask.sum(dim=1)`。
这样 padding 和 prompt 部分自动被排除。

### mask 的两个职责

1. **屏蔽 prompt**：只有模型自己生成的 token 才参与策略梯度，输入部分不训练
2. **屏蔽 padding**：同 batch 内响应不等长，右侧 padding 必须排除

In [ ]:
def build_completion_mask(seq_len, prompt_len, completion_lens, dtype=torch.float32):
    """
    构建 completion mask：标记哪些位置是「模型生成的有效 token」

    Args:
        seq_len:         shift 后的序列长度（= 原始 seq_len - 1）
        prompt_len:      prompt 在 shift 后占据的长度
        completion_lens: List[int] 或 Tensor[G]，每条响应各自的有效生成长度
    Returns:
        Tensor[G, seq_len]，1 表示参与 loss 计算，0 表示屏蔽

    mask[i] 的布局：
        [0, 0, ..., 0,  1, 1, ..., 1,  0, 0, ..., 0]
         └─ prompt ─┘   └ completion ┘  └ padding ─┘
    """
    completion_lens = torch.as_tensor(completion_lens)
    G = len(completion_lens)
    pos = torch.arange(seq_len).unsqueeze(0)                      # [1, seq_len]
    start = prompt_len                                           # 生成部分起点
    end = prompt_len + completion_lens.unsqueeze(1)              # [G, 1] 各自的终点
    return ((pos >= start) & (pos < end)).to(dtype)


def grpo_loss(pi_logprob, pi_old_logprob, pi_ref_logprob, rewards,
              completion_mask, epsilon=0.2, beta=0.01, is_debug=False):
    """
    GRPO Loss（DeepSeekMath arXiv:2402.03300 公式 3）

        L = -(1/G) * sum_i (1/|o_i|) * sum_t [ min(r*A, clip(r,1-e,1+e)*A) - beta*KL ]

    Args:
        pi_logprob:      当前策略 log prob，[G, seq_len]
        pi_old_logprob:  采样时策略（rollout 用的那份）log prob，[G, seq_len]
        pi_ref_logprob:  参考 SFT 模型 log prob，[G, seq_len]
        rewards:         每条响应的标量奖励，[G]
        completion_mask: 有效 token 掩码，[G, seq_len]
        epsilon:         PPO 裁剪阈值，论文默认 0.2
        beta:            KL 惩罚系数，论文默认 0.04（社区常用 0.001~0.01）
    Returns:
        标量 loss（已可直接 backward）
    """
    # ============================================================
    # Step 1: 组相对优势 —— GRPO 去掉 Critic 的关键
    # 同一条响应的所有 token 共享同一个 A_i（outcome supervision），
    # 故 [G] unsqueeze 成 [G, 1] 后广播到全部 token 位置
    # ============================================================
    advantage = grpo_advantage(rewards).unsqueeze(dim=1)          # [G] -> [G, 1]

    # ============================================================
    # Step 2: 重要性采样比率 r = pi_theta / pi_theta_old
    # 用 log 差再 exp，等价于直接相除但数值稳定（避免两个极小概率相除）
    # ============================================================
    ratio = torch.exp(pi_logprob - pi_old_logprob)                # [G, seq_len]

    # ============================================================
    # Step 3: 裁剪比率到信任域 [1-epsilon, 1+epsilon]
    # 防止单步更新把策略推得离采样分布太远，导致重要性采样失效
    # ============================================================
    ratio_clip = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)

    # ============================================================
    # Step 4: 取 min —— 悲观下界
    # A>0 时：限制 ratio 上界，防止对好样本过度激励
    # A<0 时：限制 ratio 下界，防止对差样本过度惩罚
    # 详细的分段分析见下方「clip 机制解析」一节
    # ============================================================
    policy_gradient = torch.minimum(ratio * advantage, ratio_clip * advantage)

    # ============================================================
    # Step 5: KL 惩罚（k3 估计量）
    # 与 PPO 不同，GRPO 把 KL 直接作为 loss 的一项，而非塞进 reward
    # ============================================================
    kl = grpo_kl(pi_logprob, pi_ref_logprob)                      # [G, seq_len]

    # ============================================================
    # Step 6: 逐 token 目标 = 策略梯度项 - beta * KL
    # ============================================================
    per_token_obj = policy_gradient - beta * kl                   # [G, seq_len]

    # ============================================================
    # Step 7: 两级归一化（GRPO 的 sequence-level 归一化）
    #   内层 1/|o_i|：每条响应内部对 token 取均值
    #   外层 1/G    ：组内对序列取均值
    # |o_i| 从 mask 求得，天然支持不等长响应（不要用标量广播！）
    # clamp(min=1) 防止空响应导致除零
    # ============================================================
    len_oi = completion_mask.sum(dim=1).clamp(min=1)              # [G]
    per_seq_obj = (per_token_obj * completion_mask).sum(dim=1) / len_oi   # [G]

    # ============================================================
    # Step 8: 取负，把「最大化目标 J」转成「最小化损失 L」
    # ============================================================
    loss = -per_seq_obj.mean()

    if is_debug:
        m = completion_mask.bool()
        print(f"[Rewards]    {rewards.tolist()}")
        print(f"[Advantage]  {[round(v, 4) for v in advantage.squeeze(1).tolist()]}")
        print(f"[|o_i|]      {len_oi.tolist()}")
        print(f"[Ratio]      min={ratio[m].min():.4f}  max={ratio[m].max():.4f}  "
              f"被裁剪比例={(ratio[m] != ratio_clip[m]).float().mean():.1%}")
        print(f"[KL]         mean={kl[m].mean():.6f}  min={kl[m].min():.3e} (必须>=0)")
        print(f"[Loss]       {loss.item():.6f}")

    return loss

In [ ]:
# ══════════════════ 端到端测试 ══════════════════
torch.manual_seed(0)

G, seq_len, vocab_size = 3, 6, 32
prompt_len = 3          # 前 3 个 token 是 prompt

# --- 模拟三个模型的输出分布 ---
pi_logits     = torch.randn(G, seq_len, vocab_size)                  # 当前策略
pi_old_logits = pi_logits + torch.randn(G, seq_len, vocab_size) * 0.1  # 旧策略（略有差异）
pi_ref_logits = pi_logits + torch.randn(G, seq_len, vocab_size) * 0.3  # 参考模型

# --- 同一 prompt 采样出的 G 条响应（前 3 个 token 相同 = 同一 prompt）---
token_ids = torch.tensor([[11, 12, 13, 14, 15, 16],
                          [11, 12, 13, 15, 16, 17],
                          [11, 12, 13, 16, 17, 18]])

# --- 取 log prob（内部完成 shift 对齐，长度变为 seq_len-1 = 5）---
pi_logprob     = gather_token_logprob(pi_logits,     token_ids)
pi_old_logprob = gather_token_logprob(pi_old_logits, token_ids)
pi_ref_logprob = gather_token_logprob(pi_ref_logits, token_ids)
print(f"shift 后 logprob shape: {tuple(pi_logprob.shape)}  (seq_len {seq_len} → {seq_len-1})\n")

# --- 构建 mask：shift 后 prompt 占 prompt_len-1=2 列，三条响应长度分别为 3/2/1 ---
completion_mask = build_completion_mask(seq_len - 1, prompt_len - 1, [3, 2, 1])
print("completion_mask（0=屏蔽 prompt/padding，1=参与训练）:")
print(completion_mask, "\n")

# --- 奖励：第 0、2 条答对，第 1 条答错 ---
rewards = torch.tensor([1.0, 0.0, 1.0])

loss = grpo_loss(pi_logprob, pi_old_logprob, pi_ref_logprob,
                 rewards, completion_mask, epsilon=0.2, beta=0.01, is_debug=True)

# --- 反向传播可用性检查 ---
pi_logits.requires_grad_(True)
lp = gather_token_logprob(pi_logits, token_ids)
grpo_loss(lp, pi_old_logprob, pi_ref_logprob, rewards, completion_mask).backward()
print(f"\n[梯度检查]   grad norm = {pi_logits.grad.norm():.6f}  (非零即可正常训练)")

In [ ]:
# ══════════════════ 关键性质验证 ══════════════════

# 性质 1：ratio=1、KL=0 时，loss 应精确等于 -mean(A_i)
#         同时验证不等长响应的 1/|o_i| 归一化是否正确
zero = torch.zeros(3, 5)
mask = torch.tensor([[0., 1, 1, 1, 1],    # |o_1| = 4
                     [0., 0, 1, 1, 0],    # |o_2| = 2
                     [0., 0, 0, 1, 0]])   # |o_3| = 1
rew  = torch.tensor([1.0, 0.0, 1.0])
loss = grpo_loss(zero, zero, zero, rew, mask)
expect = -grpo_advantage(rew).mean()
print(f"[不等长归一化] 响应长度 {mask.sum(1).tolist()}")
print(f"               loss={loss:.6f}  期望={expect:.6f}  ✅" )
assert torch.allclose(loss, expect, atol=1e-6)

# 性质 2：组内全对/全错 → 优势全零 → 无梯度（DAPO 要解决的痛点）
m4 = torch.ones(4, 5); m4[:, 0] = 0
z4 = torch.zeros(4, 5)
for r in [torch.ones(4), torch.zeros(4)]:
    l = grpo_loss(z4, z4, z4, r, m4)
    print(f"[退化情形]     奖励全为 {r[0]:.0f} → loss = {l:.3e}  (梯度为零，样本浪费)")
    assert abs(l) < 1e-6

# 性质 3：梯度方向 —— 正优势样本的 logprob 应被推高
p = torch.zeros(2, 3, requires_grad=True)
r = torch.tensor([1.0, 0.0])
grpo_loss(p, torch.zeros(2, 3), torch.zeros(2, 3), r, torch.ones(2, 3), beta=0.0).backward()
print(f"\n[梯度方向]     A = {[round(v,3) for v in grpo_advantage(r).tolist()]}")
print(f"               dL/d(logprob) = {[round(v,4) for v in p.grad[:,0].tolist()]}")
print(f"               正优势 → 梯度为负 → 梯度下降提升其 logprob ✅")
assert p.grad[0, 0] < 0 < p.grad[1, 0]

print("\n全部性质验证通过 ✅")

---
## 5. clip 机制解析：为什么是 `min` 而不是 `max`

把 $\min(rA,\ \text{clip}(r)A)$ 按优势正负拆开看：

**当 $\hat{A} > 0$（这是条好响应，想提高它的概率）**

$$\min(rA, \text{clip}(r)A) = A \cdot \min(r, \text{clip}(r)) = A \cdot \min(r,\ 1+\epsilon)$$

→ $r$ 涨过 $1+\epsilon$ 后目标值被**封顶**，梯度归零。防止「一次性把好样本的概率拉到天上」。

**当 $\hat{A} < 0$（这是条差响应，想降低它的概率）**

$$\min(rA, \text{clip}(r)A) = A \cdot \max(r, \text{clip}(r)) = A \cdot \max(r,\ 1-\epsilon)$$

（乘负数时 min 翻转成 max）→ $r$ 跌破 $1-\epsilon$ 后目标值被**托底**，梯度归零。防止过度打压。

**统一直觉**：`min` 让目标函数成为真实目标的**悲观下界（lower bound）**。
只在「更新方向已经走得够远」时才切断梯度，而反方向的修正永远畅通——
所以策略走错了还能拉回来，但不会一步冲太远。

In [ ]:
# ══════════ 可视化 clip 的作用区间 ══════════
# 图表标签用英文，与 dapo_loss.ipynb 保持一致，避免中文字体缺失问题
ratio_range = torch.linspace(0.5, 1.5, 400)
epsilon = 0.2

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, A, title in zip(axes, [1.0, -1.0],
                        ["Advantage A > 0  (good response, push probability UP)",
                         "Advantage A < 0  (bad response, push probability DOWN)"]):
    unclipped = ratio_range * A
    clipped   = torch.clamp(ratio_range, 1 - epsilon, 1 + epsilon) * A
    objective = torch.minimum(unclipped, clipped)

    ax.plot(ratio_range, unclipped, '--', color='gray',    label='r · A  (unclipped)')
    ax.plot(ratio_range, clipped,   '--', color='orange',  label='clip(r) · A')
    ax.plot(ratio_range, objective, '-',  color='crimson', lw=2.5,
            label='min(·, ·)  ← actual objective')
    ax.axvline(1 - epsilon, color='b', ls=':', alpha=.6, label=f'1±ε (ε={epsilon})')
    ax.axvline(1 + epsilon, color='b', ls=':', alpha=.6)
    ax.axvline(1.0, color='k', ls='-', alpha=.3)
    ax.set_title(title)
    ax.set_xlabel('ratio   r = π_θ / π_old')
    ax.set_ylabel('objective value')
    ax.legend(); ax.grid(alpha=.3)

plt.tight_layout(); plt.show()

# 梯度是否被切断
print(f"{'ratio':>7} | {'A>0 梯度':>10} | {'A<0 梯度':>10}")
print("-" * 34)
for rv in [0.7, 0.85, 1.0, 1.15, 1.3]:
    row = []
    for A in [1.0, -1.0]:
        x = torch.tensor([rv], requires_grad=True)
        torch.minimum(x * A, torch.clamp(x, 1-epsilon, 1+epsilon) * A).backward()
        row.append("有" if x.grad.abs() > 1e-9 else "切断")
    print(f"{rv:>7.2f} | {row[0]:>11} | {row[1]:>11}")
print("\n↑ A>0 时 r 过大被切断；A<0 时 r 过小被切断。反方向修正始终保留梯度。")

---
## 6. 工程实现的两个「与论文不一致」之处

### 6.1 TRL 中 clip 常常不起作用

TRL 的 `GRPOTrainer` 在 `num_iterations == 1`（即每批 rollout 只更新一次，无 mini-batch 内循环）时：

```python
old_per_token_logps = per_token_logps.detach()      # 就是当前策略自己
ratio = torch.exp(per_token_logps - old_per_token_logps)   # 恒等于 1
```

$\pi_{\theta_{\text{old}}}$ 就是 $\pi_\theta$ 本身（只是 detach 了），所以 **ratio 恒为 1，永远落在裁剪区间内**，
`clip` 完全是空操作。此时 GRPO 退化成带 baseline 的 REINFORCE。

只有 `num_iterations > 1`（同一批数据反复更新多次，off-policy）时 clip 才真正生效。

> `detach()` 虽然让 ratio 数值恒为 1，但它**仍然携带梯度**：
> $\nabla_\theta \exp(\log\pi_\theta - \text{const}) = \exp(\cdot) \cdot \nabla_\theta \log\pi_\theta = \nabla_\theta\log\pi_\theta$，
> 恰好还原成 REINFORCE 的 $\nabla\log\pi \cdot A$。这是个常用技巧，不是 bug。

### 6.2 std 的有偏 / 无偏

论文写 $\text{std}(\mathbf{r})$ 指总体标准差（分母 $n$），PyTorch 的 `.std()` 默认无偏（分母 $n-1$）。
本 notebook 沿用 PyTorch 默认值以对齐 verl / TRL 的实际实现，$G \geq 8$ 时差异可忽略。

In [ ]:
# ══════════ 验证：detach 让 ratio 恒为 1，但梯度仍然存在 ══════════
logprob = torch.tensor([-0.6931], requires_grad=True)   # log(0.5)

old_logprob = logprob.detach()          # TRL 在 num_iterations=1 时的做法
ratio = torch.exp(logprob - old_logprob)
print(f"ratio 数值 = {ratio.item():.6f}   ← 恒为 1，clip(1, 0.8, 1.2) 不做任何事")

A = torch.tensor([2.0])
(ratio * A).backward()
print(f"d(ratio·A)/d(logprob) = {logprob.grad.item():.6f}")
print(f"REINFORCE 的梯度 A     = {A.item():.6f}   ← 两者相同")
print("\n结论：ratio 恒为 1 时 GRPO 退化为「带 group baseline 的 REINFORCE」，")
print("      clip 机制只有在 num_iterations > 1（off-policy 复用数据）时才真正生效。")

---
## 7. 本 notebook 修正记录

原始实现存在以下问题，均已修正：

| # | 位置 | 问题 | 修正 |
|---|------|------|------|
| 1 | `grpo_kl` | 写成 `pi_logprob.exp() - (ref-pi) - 1`，第一项应为 `exp(ref-pi)`。导致 KL 可为负、且同分布时不为 0 | 改为 `exp(log_r) - log_r - 1`，并加非负性 / 无偏性验证 |
| 2 | gather logprob | 未做 causal LM 的 shift 对齐，取到 $\log\pi(x_t \mid x_{\leq t})$（自己预测自己） | 抽出 `gather_token_logprob`，用 `logits[:, :-1]` 配 `token_ids[:, 1:]` |
| 3 | `len_oi` | 用标量长度广播到整个 batch，假设所有响应等长 | 从 `completion_mask.sum(dim=1)` 求得，支持不等长 |
| 4 | mask | 只区分 prompt/completion，未处理 padding | 抽出 `build_completion_mask`，同时屏蔽 prompt 与 padding |
| 5 | 超参 | `epsilon` / `beta` 硬编码在函数体内 | 提升为带默认值的函数参数 |
| 6 | 测试 | 传入未归一化的 int64 原始 reward 当作 advantage | 改为传 `rewards`，函数内部调用 `grpo_advantage` |

## 8. 小结

**GRPO 三个记忆点**

1. **去 Critic**：组内 z-score 当优势，省掉一个同等规模的 Value Network
2. **共享优势**：整条响应的所有 token 共享同一个 $\hat{A}_i$（outcome supervision）
3. **KL 在 loss 上**：不像 PPO 塞进 reward，梯度路径更短

**两个已知短板 → 催生了后续算法**

| 短板 | 后续方案 |
|------|----------|
| 组内全对/全错 → 优势为 0，样本浪费 | **DAPO** 的 Dynamic Sampling |
| 对称 clip 压制低概率 token → 熵坍缩 | **DAPO** 的 Clip-Higher |
| sequence-level 归一化对长响应不友好 | **DAPO** 的 Token-Level Loss |
| token 级 IS 比率方差大、与序列级奖励不匹配 | **GSPO** 的序列级重要性采样 |
| clip 直接丢弃梯度导致信息损失 | **CISPO** 的 IS 权重裁剪（保留全部 token 梯度） |

推荐按 `grpo → dapo → gspo → cispo → gigpo` 的顺序阅读本系列。
另见同目录 `grpo_analysis.ipynb`：分析 GRPO loss 为何常为负值、以及为何训练中会「上升」。